In [ ]:
import sys
!{sys.executable} -m pip install biopython
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import molecular_weight
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils import gc_fraction
import math

# ===== PART 1: GC Content Calculation =====
print("=" * 60)
print("PART 1: GC Content Calculation")
print("=" * 60)

# Create sample DNA sequences
dna_seq1 = Seq("ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG")
dna_seq2 = Seq("AAAAAATTTTTTGGGGGGCCCCCC")
dna_seq3 = Seq("ATATATATATAT")

sequences = [
    ("High GC", dna_seq1),
    ("Balanced", dna_seq2),
    ("Low GC", dna_seq3)
]

print("\nMethod 1: Manual GC Content Calculation")
for name, seq in sequences:
    g_count = seq.count('G')
    c_count = seq.count('C')
    gc_count = g_count + c_count
    gc_percentage = (gc_count / len(seq)) * 100
    print(f"{name}: {gc_percentage:.2f}% GC ({gc_count}/{len(seq)} bases)")

print("\n\nMethod 2: Using Bio.SeqUtils")


for name, seq in sequences:
    gc_percent = 100 * gc_fraction(seq)
    print(f"{name}: {gc_percent:.2f}% GC")

# ===== PART 2: Molecular Weight =====
print("\n" + "=" * 60)
print("PART 2: Molecular Weight Calculation")
print("=" * 60)

print("\nDNA Molecular Weight:")
for name, seq in sequences:
    # Calculate molecular weight (default is double-stranded DNA)
    mw = molecular_weight(seq, seq_type='DNA')
    print(f"{name}: {mw:.2f} g/mol")

print("\n\nRNA Molecular Weight:")
rna_seq = dna_seq1.transcribe()
rna_mw = molecular_weight(rna_seq, seq_type='RNA')
print(f"RNA sequence: {rna_mw:.2f} g/mol")

print("\n\nProtein Molecular Weight:")
protein_seq = Seq("MKVLWAALLVTFLAGCAKAKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL")
protein_mw = molecular_weight(protein_seq, seq_type='protein')
print(f"Protein sequence: {protein_mw:.2f} g/mol")

# ===== PART 3: Reverse Complement =====
print("\n" + "=" * 60)
print("PART 3: Reverse Complement")
print("=" * 60)

test_seq = Seq("ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG")

print(f"\nOriginal sequence:\n{test_seq}")
print(f"\nComplement:\n{test_seq.complement()}")
print(f"\nReverse:\n{test_seq[::-1]}")
print(f"\nReverse Complement:\n{test_seq.reverse_complement()}")

# Verify reverse complement is correct
print("\n\nVerifying reverse complement:")
original = test_seq[:20]
rev_comp = original.reverse_complement()
print(f"Original (first 20 bp):     {original}")
print(f"Reverse complement:         {rev_comp}")
print(f"Rev comp of rev comp:       {rev_comp.reverse_complement()}")
print(f"Matches original? {original == rev_comp.reverse_complement()}")

# ===== PART 4: Translation =====
print("\n" + "=" * 60)
print("PART 4: Translation (DNA to Protein)")
print("=" * 60)

# Create a coding sequence (must be multiple of 3)
coding_seq = Seq("ATGAAAGGTGCTAGCTAGCTAGCTAGCTAA")  # ATG=start, TAA=stop

print(f"\nDNA sequence: {coding_seq}")
print(f"Length: {len(coding_seq)} bp")

# Transcribe to RNA
rna = coding_seq.transcribe()
print(f"\nRNA sequence: {rna}")

# Translate to protein
protein = coding_seq.translate()
print(f"Protein sequence: {protein}")

# Translate with different genetic codes
print("\n\nTranslation with different genetic codes:")
print("Standard genetic code:")
protein_standard = coding_seq.translate(table=1)
print(f"  {protein_standard}")

print("\nVertebrate mitochondrial code:")
protein_vert_mito = coding_seq.translate(table=2)
print(f"  {protein_vert_mito}")

# ===== PART 5: Finding Open Reading Frames (ORFs) =====
print("\n" + "=" * 60)
print("PART 5: Finding Open Reading Frames (ORFs)")
print("=" * 60)

def find_orfs(sequence, min_length=100):
    """Find all ORFs in a sequence"""
    orfs = []
    seq_str = str(sequence)

    # Check all three reading frames
    for frame in range(3):
        for i in range(frame, len(seq_str) - 2, 3):
            codon = seq_str[i:i+3]

            # Look for start codon
            if codon == 'ATG':
                # Look for stop codon
                for j in range(i + 3, len(seq_str) - 2, 3):
                    stop_codon = seq_str[j:j+3]
                    if stop_codon in ['TAA', 'TAG', 'TGA']:
                        orf_seq = seq_str[i:j+3]
                        if len(orf_seq) >= min_length:
                            orfs.append({
                                'sequence': orf_seq,
                                'start': i,
                                'end': j + 3,
                                'frame': frame,
                                'length': len(orf_seq)
                            })
                        break

    return orfs

# Test ORF finding
test_orf_seq = Seq("ATGAAAGGTGCTAGCTAGCTAGCTAGCTAAATGCCGCCGCCGCCGCCGTAA")
orfs = find_orfs(test_orf_seq, min_length=30)

print(f"\nSequence: {test_orf_seq}")
print(f"\nFound {len(orfs)} ORF(s):")
for i, orf in enumerate(orfs, 1):
    print(f"\nORF {i}:")
    print(f"  Position: {orf['start']}-{orf['end']}")
    print(f"  Frame: {orf['frame']}")
    print(f"  Length: {orf['length']} bp")
    print(f"  Sequence: {orf['sequence']}")
    protein = Seq(orf['sequence']).translate()
    print(f"  Protein: {protein}")

# ===== PART 6: Codon Usage Analysis =====
print("\n" + "=" * 60)
print("PART 6: Codon Usage Analysis")
print("=" * 60)

def analyze_codons(sequence):
    """Analyze codon usage in a sequence"""
    codon_counts = {}
    seq_str = str(sequence)

    # Count codons
    for i in range(0, len(seq_str) - 2, 3):
        codon = seq_str[i:i+3]
        if len(codon) == 3:
            codon_counts[codon] = codon_counts.get(codon, 0) + 1

    return codon_counts

test_codon_seq = Seq("ATGAAAGGTGCTAGCTAGCTAGCTAGCTAA")
codon_usage = analyze_codons(test_codon_seq)

print(f"\nSequence: {test_codon_seq}")
print(f"\nCodon usage:")
for codon, count in sorted(codon_usage.items()):
    print(f"  {codon}: {count}")

# ===== PART 7: Protein Analysis =====
print("\n" + "=" * 60)
print("PART 7: Protein Analysis")
print("=" * 60)

protein_seq = Seq("MKVLWAALLVTFLAGCAKAKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL")

# Use ProteinAnalysis for detailed analysis
protein_analysis = ProteinAnalysis(str(protein_seq))

print(f"\nProtein sequence length: {len(protein_seq)} amino acids")
print(f"Molecular weight: {protein_analysis.molecular_weight():.2f} g/mol")
print(f"Isoelectric point (pI): {protein_analysis.isoelectric_point():.2f}")
print(f"Aromaticity: {protein_analysis.aromaticity():.4f}")
print(f"Instability index: {protein_analysis.instability_index():.2f}")
print(f"GRAVY (hydropathy): {protein_analysis.gravy():.4f}")

# Amino acid composition
print(f"\nAmino acid composition:")
aa_composition = protein_analysis.count_amino_acids()
for aa, percent in sorted(aa_composition.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {aa}: {percent*100/len(protein_seq):.2f}%")

# Secondary structure prediction (Helix, Turn, Sheet)
helix, turn, sheet = protein_analysis.secondary_structure_fraction()
print(f"\nSecondary structure prediction:")
print(f"  Helix: {helix*100:.2f}%")
print(f"  Turn: {turn*100:.2f}%")
print(f"  Sheet: {sheet*100:.2f}%")

# ===== PART 8: Sequence Comparison =====
print("\n" + "=" * 60)
print("PART 8: Sequence Comparison")
print("=" * 60)

seq_a = Seq("ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG")
seq_b = Seq("ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG")
seq_c = Seq("ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATAA")

def calculate_similarity(seq1, seq2):
    """Calculate sequence similarity percentage"""
    if len(seq1) != len(seq2):
        return None

    matches = sum(1 for a, b in zip(seq1, seq2) if a == b)
    return (matches / len(seq1)) * 100

print(f"\nSequence A: {seq_a[:30]}...")
print(f"Sequence B: {seq_b[:30]}...")
print(f"Sequence C: {seq_c[:30]}...")

sim_ab = calculate_similarity(seq_a, seq_b)
sim_ac = calculate_similarity(seq_a, seq_c)
sim_bc = calculate_similarity(seq_b, seq_c)

print(f"\nSimilarity:")
print(f"  A vs B: {sim_ab:.2f}%")
print(f"  A vs C: {sim_ac:.2f}%")
print(f"  B vs C: {sim_bc:.2f}%")

print("\n" + "=" * 60)
print("Lab 3 Complete!")
print("=" * 60)


PART 1: GC Content Calculation

Method 1: Manual GC Content Calculation
High GC: 50.82% GC (31/61 bases)
Balanced: 50.00% GC (12/24 bases)
Low GC: 0.00% GC (0/12 bases)


Method 2: Using Bio.SeqUtils
High GC: 50.82% GC
Balanced: 50.00% GC
Low GC: 0.00% GC

PART 2: Molecular Weight Calculation

DNA Molecular Weight:
High GC: 18884.03 g/mol
Balanced: 7432.74 g/mol
Low GC: 3722.41 g/mol


RNA Molecular Weight:
RNA sequence: 19649.60 g/mol


Protein Molecular Weight:
Protein sequence: 32774.94 g/mol

PART 3: Reverse Complement

Original sequence:
ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG

Complement:
TACGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGC

Reverse:
GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCGTA

Reverse Complement:
CGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGCAT


Verifying reverse complement:
Original (first 20 bp):     ATGCGATCGATCGATCGATC
Reverse complement:         GATCGATCGATCGATCGCAT
Rev comp of rev comp:       A